In [1]:
import pandas as pd

pd.show_versions()


INSTALLED VERSIONS
------------------
commit                : d9cdd2ee5a58015ef6f4d15c7226110c9aab8140
python                : 3.12.7.final.0
python-bits           : 64
OS                    : Windows
OS-release            : 10
Version               : 10.0.19045
machine               : AMD64
processor             : Intel64 Family 6 Model 142 Stepping 10, GenuineIntel
byteorder             : little
LC_ALL                : None
LANG                  : None
LOCALE                : Korean_Korea.949

pandas                : 2.2.2
numpy                 : 1.26.4
pytz                  : 2024.1
dateutil              : 2.9.0.post0
setuptools            : 75.1.0
pip                   : 24.2
Cython                : None
pytest                : 7.4.4
hypothesis            : None
sphinx                : 7.3.7
blosc                 : None
feather               : None
xlsxwriter            : None
lxml.etree            : 5.2.1
html5lib              : None
pymysql               : None
psycopg2         

In [2]:
nsmc_train_df = pd.read_csv('ratings_train.txt', encoding = 'utf8', sep = '\t', engine = 'python')

nsmc_train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
nsmc_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [4]:
nsmc_train_df = nsmc_train_df[nsmc_train_df['document'].notnull()]
nsmc_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


In [5]:
nsmc_train_df['label'].value_counts()

0    75170
1    74825
Name: label, dtype: int64

In [6]:
import re

nsmc_train_df['document'] = nsmc_train_df['document'].apply(lambda x : re.sub(r'[^ ㄱ-ㅣ가-힣]+', " ", x))
nsmc_train_df.head()

,id,document,label
0,9976970,아 더빙 진짜 짜증나네요 목소리,0
1,3819312,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 솔직히 재미는 없다 평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [7]:
nsmc_test_df = pd.read_csv('ratings_test.txt', encoding = 'utf8', sep = '\t', engine = 'python')
nsmc_test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [8]:
nsmc_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [9]:
nsmc_test_df = nsmc_test_df[nsmc_test_df['document'].notnull()] 

In [10]:
nsmc_test_df['label'].value_counts()

1    25171
0    24826
Name: label, dtype: int64

In [11]:
nsmc_test_df['document'] = nsmc_test_df['document'].apply(lambda x : re.sub(r'[^ ㄱ-ㅣ가-힣]+', " ", x))
nsmc_test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,,0
2,8544678,뭐야 이 평점들은 나쁘진 않지만 점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임 돈주고 보기에는,0
4,6723715,만 아니었어도 별 다섯 개 줬을텐데 왜 로 나와서 제 심기를 불편하게 하죠,0


In [12]:
!pip install konlpy

In [13]:
from konlpy.tag import Okt

okt = Okt()

In [14]:
def okt_tokenizer(text):
    tokens = okt.morphs(text)
    return tokens

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(tokenizer = okt_tokenizer, ngram_range = (1, 2), min_df = 3, max_df = 0.9)
tfidf.fit(nsmc_train_df['document'])
nsmc_train_tfidf = tfidf.transform(nsmc_train_df['document'])

C:\Anaconda3\lib\site-packages\sklearn\feature_extraction\text.py:489: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn("The parameter 'token_pattern' will not be used"


In [16]:
print(type(nsmc_train_tfidf))

<class 'scipy.sparse.csr.csr_matrix'>


In [17]:
from sklearn.linear_model import LogisticRegression

SA_lr = LogisticRegression(random_state = 0)

In [18]:
SA_lr.fit(nsmc_train_tfidf, nsmc_train_df['label'])

C:\Anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=0)

In [19]:
from sklearn.model_selection import GridSearchCV

params = {'C': [1, 3, 3.5, 4, 4.5, 5]}
SA_lr_grid_cv = GridSearchCV(SA_lr, param_grid = params, cv = 3, scoring = 'accuracy', verbose = 1)

In [20]:
SA_lr_grid_cv.fit(nsmc_train_tfidf, nsmc_train_df['label'])

Fitting 3 folds for each of 6 candidates, totalling 18 fits


C:\Anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:763: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

GridSearchCV(cv=3, estimator=LogisticRegression(random_state=0),
             param_grid={'C': [1, 3, 3.5, 4, 4.5, 5]}, scoring='accuracy',
             verbose=1)

In [21]:
print(SA_lr_grid_cv.best_params_, round(SA_lr_grid_cv.best_score_, 4))

{'C': 3} 0.8553


In [22]:
SA_lr_best = SA_lr_grid_cv.best_estimator_

In [23]:
nsmc_test_tfidf = tfidf.transform(nsmc_test_df['document'])

In [24]:
test_predict = SA_lr_best.predict(nsmc_test_tfidf)

In [25]:
print(len(test_predict))

49997


In [26]:
from sklearn.metrics import accuracy_score

print('감성 분석 정확도 : ', round(accuracy_score(nsmc_test_df['label'], test_predict), 3))


감성 분석 정확도 :  0.858


In [27]:
#네이버 에르메스 분석

In [29]:
import json

file_name = '에르메스_naver_news'
with open(file_name + '.json', encoding = 'utf8') as j_f:
    data = json.load(j_f)

In [30]:
print(data)

[{'cnt': 1, 'description': '<b>에르메스</b>와 롤렉스는 매년 한 차례 씩 가격을 인상해왔다. 일각에서는 리오프닝(경제활동 재개)으로 꺾인 명품 소비심리에 가격 인상을 자제할 것이라는 시각도 있다. 산업통상자원부에 따르면 올해 4월 백화점... ', 'link': 'https://n.news.naver.com/mnews/article/011/0004063808?sid=101', 'org_link': 'https://www.sedaily.com/NewsView/2677CAEZE0', 'pDate': '2022-06-11 08:01:00', 'title': '&quot;결혼식 교복이냐&quot; 굴욕 겪은 샤넬…또 가격 인상설 솔솔'}, {'cnt': 2, 'description': '이전에도 삼성과 톰브라운, 애플과 <b>에르메스</b>의 스마트워치 협업 소식은 있었습니다. 인기있는 브랜드 혹은 디자인 능력을 가진 명품 브랜드, 혹은 팬덤을 가진 남성 그룹 방탄소년단 등과의 협업 등 테크업계서도... ', 'link': 'https://n.news.naver.com/mnews/article/081/0003279307?sid=105', 'org_link': 'https://www.seoul.co.kr/news/newsView.php?id=20220610500148&wlog_tag3=naver', 'pDate': '2022-06-10 20:57:00', 'title': '너도 나도 팔목에 스마트워치…결국 ‘이 브랜드’ 관심 끌었다 [명품톡+]'}, {'cnt': 3, 'description': "루이비통 제공 \xa0<b>에르메스</b>, 구찌, 디올 등 콧대 높은 명품 브랜드들이 국내에서 레스토랑을 열고 체험형... 참고로 국내에서 가장 먼저 체험형 마케팅 나선 곳은 '<b>에르메스</b>'다. 2006년 도산파크 지하에 '카페 마당'을... ", 'link': 'http://www.sporbiz.co.kr/news/articleView.

In [31]:
data_title = []
data_description = []
for item in data:
    data_title.append(item['title'])
    data_description.append(item['description'])

data_df = pd.DataFrame({'title':data_title, 'description':data_description})

In [32]:
data_df.head()

,title,description
0,&quot;결혼식 교복이냐&quot; 굴욕 겪은 샤넬…또 가격 인상설 솔솔,<b>에르메스</b>와 롤렉스는 매년 한 차례 씩 가격을 인상해왔다. 일각에서는 리...
1,너도 나도 팔목에 스마트워치…결국 ‘이 브랜드’ 관심 끌었다 [명품톡+],"이전에도 삼성과 톰브라운, 애플과 <b>에르메스</b>의 스마트워치 협업 소식은 있..."
2,구찌·디올에 이어 세계 최초 루이비통 레스토랑까지... 왜 '코리아'일까,"루이비통 제공 <b>에르메스</b>, 구찌, 디올 등 콧대 높은 명품 브랜드들이 ..."
3,"[유통 이슈] 헨켈홈케어코리아 브레프-패스트파이브, ‘상쾌한 화장실 조성 캠...","또, <b>에르메스</b>, 구찌, 랑콤, 디올, 다이슨 등 명품 브랜드 방송의 메..."
4,"'10분 진료비가 9만원' 오은영 박사가 타고 다니는 차, 차원이 다른 수준이다","“가격에 입틀막” 오은영, <b>에르메스</b> VVIP 논란 이어 '초고가 명품 ..."


In [33]:
data_title

['&quot;결혼식 교복이냐&quot; 굴욕 겪은 샤넬…또 가격 인상설 솔솔',
 '너도 나도 팔목에 스마트워치…결국 ‘이 브랜드’ 관심 끌었다 [명품톡+]',
 "구찌·디올에 이어 세계 최초 루이비통 레스토랑까지... 왜 '코리아'일까",
 '[유통 이슈] 헨켈홈케어코리아 브레프-패스트파이브, ‘상쾌한 화장실 조성 캠...',
 "'10분 진료비가 9만원' 오은영 박사가 타고 다니는 차, 차원이 다른 수준이다",
 "패션업계가 '느림의 미학'에 빠진 이유",
 '모피로 덮인 <b>에르메스</b> 버킨백 NFT, 합법일까 위법일까[김윤희의 지식재산권 산...',
 '리바이스 “2027년까지 연 6~7% 성장…100억 달러 달성”',
 "600만원 디올백 대신 2만원 라떼…요즘 뜨는 '인증샷' 성지",
 "<b>에르메스</b>, 루이뷔통…'명품에 환장한 중국', 봉쇄 이후 보복소비",
 '샤넬 주얼리도 올랐다…평균 10% 가격 인상',
 '세 남자의 일탈 드라이브',
 '‘킴 카다시인과 도플갱어’ 24살 모델, “내가 카녜이 웨스트와 결별? 가짜뉴...',
 "정재일, 세계적 음악레이블 '데카'와 글로벌 계약",
 "30년 1위 자리 폴로에 내줬다…'토종 패션' 빈폴의 위기[박동휘의 컨슈머 리포...",
 "[뉴리테일 &amp; 더현대 ①] 유통 불모지 입성한 더현대서울, '취향 쪼개기'로 진...",
 '[Men’s Look] 셔츠 입기 좋은 지금',
 '쌓여가는 대중 브랜드 재고…美\xa0패션시장, 명품만 잘 팔린다',
 '[기획] 교촌 1991‧ GS25에 담긴 비밀은?',
 '<b>에르메스</b> 뷰티의 새로운 비전',
 '44살 카녜이 웨스트, ‘킴 카다시안 도플갱어’ 24살 여친과 5개월만에 결별[...',
 '트렌비, 명품 리셀 서비스 약 14만40000건, 총 거래액 약 1690억원 돌파',
 "'싱글맘' 박연수, <b>에르메스</b> 가방에 칼 댔다…&quot;세상 하나밖에 없어&quot;",
 '박연수, <b>에르메스</b>백 직접 리폼..&

In [34]:

data_title_tfidf = tfidf.transform(data_df['title'])

data_title_predict = SA_lr_best.predict(data_title_tfidf)

data_df['title_label'] = data_title_predict

In [35]:

data_description_tfidf = tfidf.transform(data_df['description'])

data_description_predict = SA_lr_best.predict(data_description_tfidf)

data_df['description_label'] = data_description_predict

In [36]:
data_df.head()

,title,description,title_label,description_label
0,&quot;결혼식 교복이냐&quot; 굴욕 겪은 샤넬…또 가격 인상설 솔솔,<b>에르메스</b>와 롤렉스는 매년 한 차례 씩 가격을 인상해왔다. 일각에서는 리...,0,1
1,너도 나도 팔목에 스마트워치…결국 ‘이 브랜드’ 관심 끌었다 [명품톡+],"이전에도 삼성과 톰브라운, 애플과 <b>에르메스</b>의 스마트워치 협업 소식은 있...",1,1
2,구찌·디올에 이어 세계 최초 루이비통 레스토랑까지... 왜 '코리아'일까,"루이비통 제공 <b>에르메스</b>, 구찌, 디올 등 콧대 높은 명품 브랜드들이 ...",0,1
3,"[유통 이슈] 헨켈홈케어코리아 브레프-패스트파이브, ‘상쾌한 화장실 조성 캠...","또, <b>에르메스</b>, 구찌, 랑콤, 디올, 다이슨 등 명품 브랜드 방송의 메...",0,1
4,"'10분 진료비가 9만원' 오은영 박사가 타고 다니는 차, 차원이 다른 수준이다","“가격에 입틀막” 오은영, <b>에르메스</b> VVIP 논란 이어 '초고가 명품 ...",1,0


In [37]:
data_df.to_csv(file_name+'.csv', encoding = 'utf-8')

In [38]:
data_df.head()

,title,description,title_label,description_label
0,&quot;결혼식 교복이냐&quot; 굴욕 겪은 샤넬…또 가격 인상설 솔솔,<b>에르메스</b>와 롤렉스는 매년 한 차례 씩 가격을 인상해왔다. 일각에서는 리...,0,1
1,너도 나도 팔목에 스마트워치…결국 ‘이 브랜드’ 관심 끌었다 [명품톡+],"이전에도 삼성과 톰브라운, 애플과 <b>에르메스</b>의 스마트워치 협업 소식은 있...",1,1
2,구찌·디올에 이어 세계 최초 루이비통 레스토랑까지... 왜 '코리아'일까,"루이비통 제공 <b>에르메스</b>, 구찌, 디올 등 콧대 높은 명품 브랜드들이 ...",0,1
3,"[유통 이슈] 헨켈홈케어코리아 브레프-패스트파이브, ‘상쾌한 화장실 조성 캠...","또, <b>에르메스</b>, 구찌, 랑콤, 디올, 다이슨 등 명품 브랜드 방송의 메...",0,1
4,"'10분 진료비가 9만원' 오은영 박사가 타고 다니는 차, 차원이 다른 수준이다","“가격에 입틀막” 오은영, <b>에르메스</b> VVIP 논란 이어 '초고가 명품 ...",1,0


In [39]:
data_df['title_label'].value_counts()

1    513
0    487
Name: title_label, dtype: int64

In [40]:
data_df['description_label'].value_counts()

1    672
0    328
Name: description_label, dtype: int64

In [41]:
#네이버 루이비통

In [42]:
import json

file_name = '루이비통_naver_news'
with open(file_name + '.json', encoding = 'utf8') as j_f:
    data = json.load(j_f)

In [43]:
data_title = []
data_description = []
for item in data:
    data_title.append(item['title'])
    data_description.append(item['description'])

data_df = pd.DataFrame({'title':data_title, 'description':data_description})

In [44]:

data_title_tfidf = tfidf.transform(data_df['title'])

data_title_predict = SA_lr_best.predict(data_title_tfidf)

data_df['title_label'] = data_title_predict

In [45]:

data_description_tfidf = tfidf.transform(data_df['description'])

data_description_predict = SA_lr_best.predict(data_description_tfidf)

data_df['description_label'] = data_description_predict

In [46]:
data_df.to_csv(file_name+'.csv', encoding = 'utf-8')

In [47]:
data_df['title_label'].value_counts()

1    505
0    495
Name: title_label, dtype: int64

In [48]:
data_df['description_label'].value_counts()

1    652
0    348
Name: description_label, dtype: int64

In [49]:
#네이버 샤넬

In [50]:
import json

file_name = '샤넬_naver_news'
with open(file_name + '.json', encoding = 'utf8') as j_f:
    data = json.load(j_f)

In [51]:
data_title = []
data_description = []
for item in data:
    data_title.append(item['title'])
    data_description.append(item['description'])

data_df = pd.DataFrame({'title':data_title, 'description':data_description})

In [52]:

data_title_tfidf = tfidf.transform(data_df['title'])

data_title_predict = SA_lr_best.predict(data_title_tfidf)

data_df['title_label'] = data_title_predict

In [53]:
data_description_tfidf = tfidf.transform(data_df['description'])

data_description_predict = SA_lr_best.predict(data_description_tfidf)

data_df['description_label'] = data_description_predict

In [54]:
data_df.to_csv(file_name+'.csv', encoding = 'utf-8')

In [55]:
data_df['title_label'].value_counts()

1    501
0    499
Name: title_label, dtype: int64

In [56]:
data_df['description_label'].value_counts()

1    569
0    431
Name: description_label, dtype: int64

In [57]:
#네이버 디올

In [58]:
import json

file_name = '디올_naver_news'
with open(file_name + '.json', encoding = 'utf8') as j_f:
    data = json.load(j_f)

In [59]:
data_title = []
data_description = []
for item in data:
    data_title.append(item['title'])
    data_description.append(item['description'])

data_df = pd.DataFrame({'title':data_title, 'description':data_description})

In [60]:

data_title_tfidf = tfidf.transform(data_df['title'])

data_title_predict = SA_lr_best.predict(data_title_tfidf)

data_df['title_label'] = data_title_predict

In [61]:
data_description_tfidf = tfidf.transform(data_df['description'])

data_description_predict = SA_lr_best.predict(data_description_tfidf)

data_df['description_label'] = data_description_predict

In [62]:
data_df.to_csv(file_name+'.csv', encoding = 'utf-8')

In [63]:
data_df['title_label'].value_counts()

1    540
0    460
Name: title_label, dtype: int64

In [64]:
data_df['description_label'].value_counts()

1    723
0    277
Name: description_label, dtype: int64